# Vision-Based Litter Severity (YOLO11s + LSI)

**Goal:** Train a single-class **litter** detector and compute a **Litter Severity Index (LSI)** from detections (count, area coverage, spatial spread).

**Before you run:** *Runtime → Change runtime type → **GPU** (T4/L4/A100)*.

This notebook mirrors the local repo layout: `dataset/`, `config/lsi_config.yaml`, `scripts/`, `models/`, `results/`.

## 1) Verify GPU
Ultralytics will use CUDA automatically when `torch.cuda.is_available()` is true.

In [ ]:
import torch
# True means Colab gave you a GPU and PyTorch can see it.
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    # Device name helps you document hardware in the thesis.
    print("device:", torch.cuda.get_device_name(0))

## 2) Install dependencies
`ultralytics` pulls compatible `torch` wheels on Colab; the line below is usually enough.

In [ ]:
!pip install -q ultralytics opencv-python matplotlib pyyaml

import ultralytics
# Print versions for reproducibility (cite in your methodology chapter).
print("ultralytics:", ultralytics.__version__)

## 3) Project paths on Colab
We work under `/content/litter_severity_detection` so local `README.md` commands still make sense.

In [ ]:
from pathlib import Path

ROOT = Path("/content/litter_severity_detection")
ROOT.mkdir(parents=True, exist_ok=True)
for sub in ["dataset", "models", "results", "runs", "config", "scripts"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 4) Bring your dataset
**Option A — Roboflow API (recommended):** paste workspace/project/version + API key.

**Option B:** upload a zip of your YOLO dataset and unpack to `ROOT/dataset`.

Your `data.yaml` must list `train`, `val`, `test` image folders (Ultralytics format).

In [ ]:
# --- Option A: Roboflow download (uncomment and fill) ---
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# version = project.version(YOUR_VERSION)
# dataset = version.download("yolov8", location=str(ROOT / "dataset"))
# print(dataset.location)

# --- Option B: upload zip manually ---
# from google.colab import files
# uploaded = files.upload()  # select dataset.zip
# !unzip -q dataset.zip -d $ROOT/dataset

print("Place images under:", ROOT / "dataset")

## 5) Write `data.yaml` (if Roboflow paths differ)
Open your downloaded `data.yaml` and ensure paths are correct **relative to `dataset/`**.

Typical Roboflow export already contains `train: train/images`, `val: valid/images`, etc.

In [ ]:
from pathlib import Path

data_yaml = ROOT / "dataset" / "data.yaml"
assert data_yaml.is_file(), f"Missing {data_yaml} — fix unzip paths or Roboflow download."
print(data_yaml.read_text()[:800])

## 6) Train YOLO11 **Small**
`yolo11s.pt` is the official Ultralytics **small** checkpoint (downloads on first use).

Training writes to `runs/litter_yolo11s/` and saves `weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
train_results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(ROOT / "runs"),
    name="litter_yolo11s",
    patience=50,
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

best = ROOT / "runs" / "litter_yolo11s" / "weights" / "best.pt"
assert best.is_file(), "Training did not produce best.pt"

import shutil
shutil.copy2(best, ROOT / "models" / "best.pt")
print("Saved:", ROOT / "models" / "best.pt")

## 7) Validate (mAP, precision, recall)
Use these metrics in your **Results** chapter; LSI is an **application-level** score on top of detection quality.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(ROOT / "models" / "best.pt"))
metrics = model.val(data=str(data_yaml), imgsz=640)
print(metrics)

## 8) LSI demo on one image (inline)
This cell duplicates the logic in `scripts/calculate_lsi.py` / `scripts/detect.py` so Colab is self-contained.
For publication runs, copy the `litter_severity_detection/scripts/` folder into `ROOT` and `import` them.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml
from ultralytics import YOLO

# --- Minimal LSI helpers (same idea as scripts/calculate_lsi.py) ---

def mean_pairwise_distance(pts: np.ndarray) -> float:
    n = pts.shape[0]
    if n < 2:
        return 0.0
    acc = 0.0
    pairs = 0
    for i in range(n):
        for j in range(i + 1, n):
            acc += float(np.linalg.norm(pts[i] - pts[j]))
            pairs += 1
    return acc / max(pairs, 1)


def compute_lsi_inline(xyxy, w, h, cap=25, area_scale=100.0, spread_den=0.35):
    xyxy = np.asarray(xyxy, dtype=np.float64).reshape(-1, 4)
    img_area = float(max(1, w) * max(1, h))
    n = int(xyxy.shape[0])
    if n == 0:
        return 0.0, {"count": 0, "count_score": 0, "area_score": 0, "spread_score": 0}

    areas = np.maximum(0, xyxy[:, 2] - xyxy[:, 0]) * np.maximum(0, xyxy[:, 3] - xyxy[:, 1])
    coverage = float(np.sum(areas) / img_area)

    cx = (xyxy[:, 0] + xyxy[:, 2]) * 0.5
    cy = (xyxy[:, 1] + xyxy[:, 3]) * 0.5
    pts = np.stack([cx, cy], axis=1)
    mpd = mean_pairwise_distance(pts)
    diag = float(np.hypot(w, h))

    c_score = float(min(100.0, (n / cap) * 100.0))
    a_score = float(min(100.0, max(0.0, coverage * area_scale)))
    s_score = float(min(100.0, (mpd / (diag * spread_den)) * 100.0)) if diag > 0 else 0.0

    lsi = 0.5 * c_score + 0.3 * a_score + 0.2 * s_score
    lsi = float(min(100.0, max(0.0, lsi)))
    return lsi, {
        "count": n,
        "count_score": c_score,
        "area_score": a_score,
        "spread_score": s_score,
        "coverage_fraction": coverage,
        "mean_pairwise_px": mpd,
    }


def severity(lsi: float) -> str:
    if lsi <= 30:
        return "LOW"
    if lsi <= 60:
        return "MEDIUM"
    return "HIGH"


# Pick one test image
test_dir = ROOT / "dataset" / "test" / "images"
assert test_dir.is_dir(), "Create dataset/test/images or change path"
img_path = sorted(test_dir.glob("*.*"))[0]

bgr = cv2.imread(str(img_path))
H, W = bgr.shape[:2]

model = YOLO(str(ROOT / "models" / "best.pt"))
res = model.predict(source=bgr, conf=0.25, iou=0.45, imgsz=640, verbose=False)[0]
if res.boxes is None or len(res.boxes) == 0:
    xyxy = np.zeros((0, 4))
    confs = np.zeros((0,))
else:
    xyxy = res.boxes.xyxy.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()

lsi, parts = compute_lsi_inline(xyxy, W, H)
print("LSI:", round(lsi, 2), severity(lsi), parts)

# Draw
vis = bgr.copy()
for i in range(xyxy.shape[0]):
    x1, y1, x2, y2 = map(int, xyxy[i].tolist())
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 165, 255), 2)
    cv2.putText(
        vis,
        f"{confs[i]:.2f}",
        (x1, max(0, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )

cv2.putText(
    vis,
    f"LSI {lsi:.1f} ({severity(lsi)})",
    (12, 28),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.9,
    (60, 255, 60),
    2,
    cv2.LINE_AA,
)

out = ROOT / "results" / f"colab_demo_{img_path.stem}.jpg"
cv2.imwrite(str(out), vis)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"LSI = {lsi:.1f} ({severity(lsi)})")
plt.axis("off")
plt.show()

with (out.with_suffix(".json")).open("w") as f:
    json.dump({"lsi": lsi, "severity": severity(lsi), "parts": parts}, f, indent=2)
print("Saved:", out)

## 9) Research extensions (discussion prompts)

- **Temporal tracking:** smooth LSI with exponential moving average; alert on sustained HIGH.
- **Policy layer:** map severity to recommended interventions (cleanup crew routing).
- **Hotspots:** geocode bins; aggregate weekly LSI for municipal dashboards.
- **Edge deployment:** export `best.pt` → ONNX/TensorRT; run on Jetson or Raspberry Pi.
- **Fairness & ethics:** document failure cases (night, rain, cultural litter types).

Zip `runs/`, `models/`, and `results/` before downloading from Colab.